# ML Pipeline - Inspection Outcome Prediction - AND 3 - Task 6

This notebook implements a machine learning pipeline to predict elevator inspection outcomes using the feature matrix generated in Task 5.

The goal is to:
- Establish a baseline score
- Train multiple models
- Apply feature selection
- Compare performance
- Select the best model

In [127]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("../data/feature_matrix.csv")

print("Shape:", df.shape)
df.head()

Shape: (132212, 20)


,ElevatingDevicesNumber,Latest_INSPECTION_Date,target,prior_inspection_count,prior_fail_count,prior_pass_count,prior_pass_rate,days_since_last_inspection,prior_inspection_frequency,last_inspection_outcome,prior_order_count,max_prior_riskscore,mean_prior_riskscore,prior_overdue_order_count,prior_unresolved_order_count,distinct_directive_count,device_type_encoded,insp_type_Followup,insp_type_Other,insp_type_Periodic
0,8,2012-03-05,1,0,0,0,0.0,NaN,NaN,NaN,0,NaN,NaN,0,0,0,6,0,0,1
1,8,2012-03-16,1,1,1,0,0.0,11.0,NaN,1.0,4,23.0,20.500000,2,0,0,6,1,0,0
2,8,2012-04-11,1,2,2,0,0.0,26.0,11.000000,1.0,4,23.0,20.500000,4,0,0,6,1,0,0
3,8,2012-10-11,1,3,3,0,0.0,183.0,18.500000,1.0,4,23.0,20.500000,4,0,0,6,0,1,0
4,8,2012-11-21,0,4,4,0,0.0,41.0,73.333333,1.0,11,23.0,19.545455,11,0,0,6,1,0,0


## Data Loading

The feature matrix generated in Task 5 is loaded and validated. This dataset contains engineered numerical features and a binary target variable used for model training.

In [128]:
print("Columns:")
print(df.columns.tolist())
target_col = "target"

Columns:
['ElevatingDevicesNumber', 'Latest_INSPECTION_Date', 'target', 'prior_inspection_count', 'prior_fail_count', 'prior_pass_count', 'prior_pass_rate', 'days_since_last_inspection', 'prior_inspection_frequency', 'last_inspection_outcome', 'prior_order_count', 'max_prior_riskscore', 'mean_prior_riskscore', 'prior_overdue_order_count', 'prior_unresolved_order_count', 'distinct_directive_count', 'device_type_encoded', 'insp_type_Followup', 'insp_type_Other', 'insp_type_Periodic']


In [129]:
# y = df[target_col]
# X = df.drop(columns=[target_col])

FEATURE_COLS = [
      'prior_inspection_count', 'prior_fail_count', 'prior_pass_count', 'prior_pass_rate',
      'days_since_last_inspection', 'prior_inspection_frequency', 'last_inspection_outcome',
      'prior_order_count', 'max_prior_riskscore', 'mean_prior_riskscore',
      'prior_overdue_order_count', 'prior_unresolved_order_count', 'distinct_directive_count',
      'device_type_encoded', 'insp_type_Followup', 'insp_type_Other', 'insp_type_Periodic'
  ]
X = df[FEATURE_COLS]
y = df["target"]

In [130]:
# print("Number of samples:", df.shape[0])
# print("Number of features:", df.shape[1] - 1)
print("Number of samples:", df.shape[0])
print("Number of features:", X.shape[1])

Number of samples: 132212
Number of features: 17


In [131]:
df.isnull().sum().sort_values(ascending=False).head(10)

mean_prior_riskscore          81063
max_prior_riskscore           81063
prior_inspection_frequency    72204
days_since_last_inspection    40954
last_inspection_outcome       40954
target                            0
Latest_INSPECTION_Date            0
ElevatingDevicesNumber            0
prior_inspection_count            0
prior_pass_rate                   0
dtype: int64

In [132]:
print("Target distribution:")
print(y.value_counts())
print("\nPercentage:")
print(y.value_counts(normalize=True))

Target distribution:
target
1    107041
0     25171
Name: count, dtype: int64

Percentage:
target
1    0.809616
0    0.190384
Name: proportion, dtype: float64


## Target Variable Analysis

The target variable represents inspection outcomes:

- 0 → PASS
- 1 → NOT PASS (failures or follow-ups)

The class distribution shows a significant imbalance (~81% NOT PASS), which makes accuracy misleading as a standalone evaluation metric.

**Primary evaluation metric: F1 macro.** It averages F1 score equally across both classes, ensuring the minority class (PASS) is not ignored. A model that predicts only NOT PASS scores 0 for class-0 F1, which F1 macro penalizes. ROC-AUC is computed as a supplementary metric to assess discrimination ability independently of threshold.

In [133]:
from sklearn.metrics import accuracy_score

most_common_class = y.value_counts().idxmax()

baseline_pred = np.full(len(y), most_common_class)

baseline_score = accuracy_score(y, baseline_pred)

print(f"Baseline accuracy: {baseline_score:.4f}")
print(f"Always predicting: {most_common_class}")

label_map = {0: "PASS", 1: "NOT PASS"}


Baseline accuracy: 0.8096
Always predicting: 1


In [134]:
baseline_proportion = y.value_counts(normalize=True).max()
print(f"Baseline (proportion): {baseline_proportion:.4f}")

Baseline (proportion): 0.8096


In [135]:
print(f"Always predicting class: {most_common_class} ({label_map[most_common_class]})")

Always predicting class: 1 (NOT PASS)


The baseline ROC-AUC is computed below after the train/test split, once `X_test` and `y_test` are available.

## Baseline Model

The baseline shows that simply predicting "NOT PASS" yields an accuracy of 80.96%, indicating a strong class imbalance.

**The primary comparison baseline is F1 macro.** A majority-class predictor achieves 0.4964 F1 macro on the test set — class-0 F1 is 0 (no PASS cases detected) and class-1 F1 is ~0.993 — because it never identifies the minority class. Any trained model must outperform this baseline using F1 macro.

The exact baseline values are computed via DummyClassifier after the train/test split (see next section). Baseline accuracy (0.8096) and baseline ROC-AUC (0.50) are reported as supplementary reference values.

## Temporal Train/Test Split

To prevent data leakage, the dataset is split using time order:

- Training set: earlier inspections
- Test set: later inspections

The column `Latest_INSPECTION_Date` is used only for sorting and splitting, and is NOT used as a feature.

This simulates a real-world scenario where future inspections are predicted using past data.

In [136]:
# Sort dataset chronologically
df = df.sort_values("Latest_INSPECTION_Date").reset_index(drop=True)

In [137]:
# Define 80/20 split
split_index = int(len(df) * 0.8)

In [138]:
# Create train and test sets (past vs future)
train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

In [139]:
date_col = "Latest_INSPECTION_Date"

X_train = train_df[FEATURE_COLS]
y_train = train_df[target_col]
X_test = test_df[FEATURE_COLS]
y_test = test_df[target_col]

In [140]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, f1_score

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
y_pred_dummy  = dummy.predict(X_test)
y_proba_dummy = dummy.predict_proba(X_test)

baseline_roc_auc   = roc_auc_score(y_test, y_proba_dummy[:, 1])
baseline_f1_macro  = f1_score(y_test, y_pred_dummy, average='macro')

print(f"Baseline (majority-class predictor on test set):")
print(f"  Accuracy:  {baseline_score:.4f}")
print(f"  F1 macro:  {baseline_f1_macro:.4f}")
print(f"  ROC-AUC:   {baseline_roc_auc:.4f}")

Baseline (majority-class predictor on test set):
  Accuracy:  0.8096
  F1 macro:  0.4964
  ROC-AUC:   0.5000


In [141]:
print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

print("\nTrain date range:")
print(train_df[date_col].min(), "→", train_df[date_col].max())

print("\nTest date range:")
print(test_df[date_col].min(), "→", test_df[date_col].max())

Train size: (105769, 17)
Test size: (26443, 17)

Train date range:
2011-01-04 → 2015-12-14

Test date range:
2015-12-14 → 2017-01-09


In [142]:
assert train_df[date_col].max() <= test_df[date_col].min(), \
      "Temporal split violated: training data overlaps test period"

In [143]:
print("Max train date:", train_df[date_col].max())
print("Min test date:", test_df[date_col].min())

Max train date: 2015-12-14
Min test date: 2015-12-14


The temporal split may include the same date at the boundary between train and test sets.

This is acceptable because:
- No future data is used in training
- The split still respects chronological order

Therefore, we validate that:
max(train_date) <= min(test_date)

## Logistic Regression Model

We start with Logistic Regression as a baseline learning model.

This model requires additional preprocessing due to:

- Missing values in several features
- Different feature scales (counts, days, ratios)
- Class imbalance in the dataset

To address these issues, we use a scikit-learn Pipeline that includes:
- Imputation (to handle missing values)
- Scaling (to normalize feature ranges)
- Class balancing (to avoid bias toward the majority class)

In [144]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [145]:
# Logistic Regression pipeline
lr_pipeline = Pipeline([
    # Step 1: Handle missing values
    ("imputer", SimpleImputer(strategy="median")),
    
    # Step 2: Normalize feature scale
    ("scaler", StandardScaler()),
    
    # Step 3: Train Logistic Regression model
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

In [146]:
# Train the model
lr_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. I

In [147]:
# Generate predictions
y_pred_lr = lr_pipeline.predict(X_test)

# Predict probabilities (needed for ROC-AUC and analysis)
y_proba_lr = lr_pipeline.predict_proba(X_test)

In [148]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

accuracy_lr  = accuracy_score(y_test, y_pred_lr)
f1_macro_lr  = f1_score(y_test, y_pred_lr, average='macro')
f1_class0_lr = f1_score(y_test, y_pred_lr, average=None)[0]
roc_auc_lr   = roc_auc_score(y_test, y_proba_lr[:, 1])

print("Logistic Regression Performance")
print(f"Accuracy:        {accuracy_lr:.4f}  (baseline: 0.8096)")
print(f"F1 macro:        {f1_macro_lr:.4f}")
print(f"F1 class 0 (PASS): {f1_class0_lr:.4f}")
print(f"ROC-AUC:         {roc_auc_lr:.4f}")

Logistic Regression Performance
Accuracy:        0.8398  (baseline: 0.8096)
F1 macro:        0.5058
F1 class 0 (PASS): 0.0995
ROC-AUC:         0.6771


In [149]:
from sklearn.metrics import classification_report

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))


Classification Report:
              precision    recall  f1-score   support

           0       0.05      0.61      0.10       382
           1       0.99      0.84      0.91     26061

    accuracy                           0.84     26443
   macro avg       0.52      0.73      0.51     26443
weighted avg       0.98      0.84      0.90     26443



### Model Interpretation

Logistic Regression serves as a simple and interpretable baseline model.

Because of class imbalance (~80% NOT PASS), a `class_weight="balanced"` strategy was used to ensure the model pays attention to the minority class.

**F1 macro is the primary evaluation metric** — it averages performance equally across both classes, penalizing models that ignore PASS cases. ROC-AUC is also reported as a supplementary metric.

The model's performance will later be compared with more complex models such as Random Forest and Gradient Boosting.

## Random Forest Model

Random Forest is a non-linear ensemble model that can capture complex patterns and feature interactions.

Unlike Logistic Regression, it does not require feature scaling.

However, missing values still need to be handled before training.

- Class imbalance is addressed using `class_weight="balanced"`, for the same reason as Logistic Regression.

This model serves as a more flexible alternative to Logistic Regression.

In [150]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [151]:
# Random Forest pipeline
rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

In [152]:
rf_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a fe

In [153]:
y_pred_rf = rf_pipeline.predict(X_test)
y_proba_rf = rf_pipeline.predict_proba(X_test)

In [154]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

accuracy_rf  = accuracy_score(y_test, y_pred_rf)
f1_macro_rf  = f1_score(y_test, y_pred_rf, average='macro')
f1_class0_rf = f1_score(y_test, y_pred_rf, average=None)[0]
roc_auc_rf   = roc_auc_score(y_test, y_proba_rf[:, 1])

print("Random Forest Performance")
print(f"Accuracy:          {accuracy_rf:.4f}  (baseline: 0.8096)")
print(f"F1 macro:          {f1_macro_rf:.4f}")
print(f"F1 class 0 (PASS): {f1_class0_rf:.4f}")
print(f"ROC-AUC:           {roc_auc_rf:.4f}")

Random Forest Performance
Accuracy:          0.8865  (baseline: 0.8096)
F1 macro:          0.5187
F1 class 0 (PASS): 0.0980
ROC-AUC:           0.7705


In [155]:
from sklearn.metrics import classification_report

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))


Classification Report:
              precision    recall  f1-score   support

           0       0.06      0.43      0.10       382
           1       0.99      0.89      0.94     26061

    accuracy                           0.89     26443
   macro avg       0.52      0.66      0.52     26443
weighted avg       0.98      0.89      0.93     26443



### Model Interpretation

Random Forest is a non-linear ensemble model that captures complex patterns and feature interactions without requiring feature scaling.

Like Logistic Regression, `class_weight="balanced"` is applied to prevent the model from exploiting the 80/20 class imbalance by predicting NOT PASS for every inspection.

**F1 macro is the primary evaluation metric** and is used to compare this model directly against Logistic Regression. ROC-AUC is also reported as a supplementary metric.

## Gradient Boosting Model

Gradient Boosting is an advanced ensemble method that builds trees sequentially.

Each new tree is trained to correct the errors of the previous ones, allowing the model to capture complex, non-linear patterns.

This model is included as a more sophisticated alternative to Random Forest.

In [156]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [157]:
# Gradient Boosting pipeline
gb_pipeline = Pipeline([
    # Step 1: Handle missing values
    ("imputer", SimpleImputer(strategy="median")),
    
    # Step 2: Train Gradient Boosting model
    ("model", GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42
    ))
])

In [158]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

gb_pipeline.fit(X_train, y_train, model__sample_weight=sample_weights)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a fe

In [159]:
y_pred_gb = gb_pipeline.predict(X_test)
y_proba_gb = gb_pipeline.predict_proba(X_test)

In [160]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

accuracy_gb  = accuracy_score(y_test, y_pred_gb)
f1_macro_gb  = f1_score(y_test, y_pred_gb, average='macro')
f1_class0_gb = f1_score(y_test, y_pred_gb, average=None)[0]
roc_auc_gb   = roc_auc_score(y_test, y_proba_gb[:, 1])

print("Gradient Boosting Performance")
print(f"Accuracy:          {accuracy_gb:.4f}  (baseline: 0.8096)")
print(f"F1 macro:          {f1_macro_gb:.4f}")
print(f"F1 class 0 (PASS): {f1_class0_gb:.4f}")
print(f"ROC-AUC:           {roc_auc_gb:.4f}")

Gradient Boosting Performance
Accuracy:          0.8463  (baseline: 0.8096)
F1 macro:          0.5092
F1 class 0 (PASS): 0.1025
ROC-AUC:           0.7456


In [161]:
from sklearn.metrics import classification_report

print("\nClassification Report:")
print(classification_report(y_test, y_pred_gb))


Classification Report:
              precision    recall  f1-score   support

           0       0.06      0.61      0.10       382
           1       0.99      0.85      0.92     26061

    accuracy                           0.85     26443
   macro avg       0.52      0.73      0.51     26443
weighted avg       0.98      0.85      0.90     26443



### Model Interpretation

Gradient Boosting builds trees sequentially — each tree corrects the errors of the previous one, allowing the model to capture complex non-linear patterns.

Unlike Logistic Regression and Random Forest, `GradientBoostingClassifier` does not support `class_weight` directly. Class imbalance was addressed by computing sample weights with `compute_sample_weight` and passing them via `model__sample_weight` during `.fit()`.

In this dataset, Random Forest achieved a higher ROC-AUC (0.7705) than Gradient Boosting (0.7456). Gradient Boosting with default parameters does not guarantee better results than Random Forest — it is more sensitive to hyperparameter choices and training time is significantly longer on large datasets.

## Feature Selection

To reduce noise and improve model performance, feature selection is applied using SelectKBest with mutual information.

This allows us to retain the most informative features while removing less relevant ones.

Each model will be evaluated before and after feature selection.

In [162]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif

In [163]:
k = max(5, int(X_train.shape[1] * 0.5))
print(f"k = {k} features selected out of {X_train.shape[1]}")

k = 8 features selected out of 17


In [164]:
lr_fs_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("select", SelectKBest(mutual_info_classif, k=k)),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

lr_fs_pipeline.fit(X_train, y_train)

y_pred_lr_fs  = lr_fs_pipeline.predict(X_test)
y_proba_lr_fs = lr_fs_pipeline.predict_proba(X_test)

accuracy_lr_fs  = accuracy_score(y_test, y_pred_lr_fs)
f1_macro_lr_fs  = f1_score(y_test, y_pred_lr_fs, average='macro')
f1_class0_lr_fs = f1_score(y_test, y_pred_lr_fs, average=None)[0]
roc_auc_lr_fs   = roc_auc_score(y_test, y_proba_lr_fs[:, 1])

print("LR + SelectKBest Performance")
print(f"Accuracy:          {accuracy_lr_fs:.4f}  (baseline: 0.8096)")
print(f"F1 macro:          {f1_macro_lr_fs:.4f}")
print(f"F1 class 0 (PASS): {f1_class0_lr_fs:.4f}")
print(f"ROC-AUC:           {roc_auc_lr_fs:.4f}")

# Show selected feature names (same for all models — selector is fit on training data)
selected_features = [FEATURE_COLS[i] for i in lr_fs_pipeline.named_steps['select'].get_support(indices=True)]
print(f"\nSelected features ({k}):", selected_features)

LR + SelectKBest Performance
Accuracy:          0.8367  (baseline: 0.8096)
F1 macro:          0.5040
F1 class 0 (PASS): 0.0978
ROC-AUC:           0.7166

Selected features (8): ['prior_inspection_count', 'prior_fail_count', 'prior_pass_count', 'prior_pass_rate', 'days_since_last_inspection', 'prior_inspection_frequency', 'prior_order_count', 'insp_type_Periodic']


In [165]:
rf_fs_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("select", SelectKBest(mutual_info_classif, k=k)),
    ("model", RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

rf_fs_pipeline.fit(X_train, y_train)

y_pred_rf_fs  = rf_fs_pipeline.predict(X_test)
y_proba_rf_fs = rf_fs_pipeline.predict_proba(X_test)

accuracy_rf_fs  = accuracy_score(y_test, y_pred_rf_fs)
f1_macro_rf_fs  = f1_score(y_test, y_pred_rf_fs, average='macro')
f1_class0_rf_fs = f1_score(y_test, y_pred_rf_fs, average=None)[0]
roc_auc_rf_fs   = roc_auc_score(y_test, y_proba_rf_fs[:, 1])

print("RF + SelectKBest Performance")
print(f"Accuracy:          {accuracy_rf_fs:.4f}  (baseline: 0.8096)")
print(f"F1 macro:          {f1_macro_rf_fs:.4f}")
print(f"F1 class 0 (PASS): {f1_class0_rf_fs:.4f}")
print(f"ROC-AUC:           {roc_auc_rf_fs:.4f}")

RF + SelectKBest Performance
Accuracy:          0.8476  (baseline: 0.8096)
F1 macro:          0.5096
F1 class 0 (PASS): 0.1025
ROC-AUC:           0.7196


In [166]:
gb_fs_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("select", SelectKBest(mutual_info_classif, k=k)),
    ("model", GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42
    ))
])

sample_weights_fs = compute_sample_weight(class_weight='balanced', y=y_train)
gb_fs_pipeline.fit(X_train, y_train, model__sample_weight=sample_weights_fs)

y_pred_gb_fs  = gb_fs_pipeline.predict(X_test)
y_proba_gb_fs = gb_fs_pipeline.predict_proba(X_test)

accuracy_gb_fs  = accuracy_score(y_test, y_pred_gb_fs)
f1_macro_gb_fs  = f1_score(y_test, y_pred_gb_fs, average='macro')
f1_class0_gb_fs = f1_score(y_test, y_pred_gb_fs, average=None)[0]
roc_auc_gb_fs   = roc_auc_score(y_test, y_proba_gb_fs[:, 1])

print("GB + SelectKBest Performance")
print(f"Accuracy:          {accuracy_gb_fs:.4f}  (baseline: 0.8096)")
print(f"F1 macro:          {f1_macro_gb_fs:.4f}")
print(f"F1 class 0 (PASS): {f1_class0_gb_fs:.4f}")
print(f"ROC-AUC:           {roc_auc_gb_fs:.4f}")

GB + SelectKBest Performance
Accuracy:          0.8094  (baseline: 0.8096)
F1 macro:          0.4896
F1 class 0 (PASS): 0.0856
ROC-AUC:           0.7148


In [167]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "Gradient Boosting"],
    "ROC-AUC (all features)":  [roc_auc_lr,    roc_auc_rf,    roc_auc_gb],
    "ROC-AUC (SelectKBest)":   [roc_auc_lr_fs, roc_auc_rf_fs, roc_auc_gb_fs],
    "F1 macro (all features)": [f1_macro_lr,   f1_macro_rf,   f1_macro_gb],
    "F1 macro (SelectKBest)":  [f1_macro_lr_fs,f1_macro_rf_fs,f1_macro_gb_fs],
    "Accuracy (all features)": [accuracy_lr,   accuracy_rf,   accuracy_gb],
    "Accuracy (SelectKBest)":  [accuracy_lr_fs,accuracy_rf_fs,accuracy_gb_fs],
})

print(results.to_string(index=False))
print(f"\nBaseline accuracy: {baseline_score:.4f}")
print(f"Baseline ROC-AUC:  {baseline_roc_auc:.4f}  (majority-class predictor — no discrimination)")

              Model  ROC-AUC (all features)  ROC-AUC (SelectKBest)  F1 macro (all features)  F1 macro (SelectKBest)  Accuracy (all features)  Accuracy (SelectKBest)
Logistic Regression                0.677086               0.716641                 0.505765                0.504028                 0.839769                0.836743
      Random Forest                0.770479               0.719559                 0.518741                0.509612                 0.886548                0.847635
  Gradient Boosting                0.745648               0.714806                 0.509194                0.489602                 0.846273                0.809364

Baseline accuracy: 0.8096
Baseline ROC-AUC:  0.5000  (majority-class predictor — no discrimination)


In [168]:
best_model_idx = results["F1 macro (all features)"].idxmax()
best_model = results.loc[best_model_idx]

print("Best model (by F1 macro, all features):")
print(best_model)

Best model (by F1 macro, all features):
Model                      Random Forest
ROC-AUC (all features)          0.770479
ROC-AUC (SelectKBest)           0.719559
F1 macro (all features)         0.518741
F1 macro (SelectKBest)          0.509612
Accuracy (all features)         0.886548
Accuracy (SelectKBest)          0.847635
Name: 1, dtype: object


## Best Model — Random Forest (all features)

**Selected model:** Random Forest with all 17 features  
**Primary metric:** F1 macro = 0.5187  
**Baseline F1 macro:** 0.4964 (majority-class predictor — improvement: +0.022)

### Why F1 macro as the selection criterion

F1 macro averages the F1 score equally across both classes. Under the 80/20 class imbalance, a model that always predicts NOT PASS reaches 80.96% accuracy while detecting zero PASS cases — accuracy hides this failure completely. F1 macro penalizes it by including class-0 F1 (which equals 0 for a majority-class predictor) in the average.

ROC-AUC is included as a supplementary metric and confirms the same winner: RF achieves 0.7705 (+0.2705 over baseline ROC-AUC of 0.50).

### Why Random Forest over Logistic Regression and Gradient Boosting

| Model | F1 macro (all features) | F1 macro (SelectKBest) | ROC-AUC (all features) |
|---|---|---|---|
| Baseline (majority class) | 0.4964 | 0.4964 | 0.5000 |
| Logistic Regression | 0.5058 | 0.5040 | 0.6771 |
| **Random Forest** | **0.5187** | **0.5096** | **0.7705** |
| Gradient Boosting | 0.5092 | 0.4896 | 0.7456 |

- **vs Logistic Regression:** RF achieves F1 macro 0.5187 vs LR's 0.5058 (+0.013). LR assumes linear decision boundaries — the relationship between inspection history and outcome is non-linear, which RF captures better through ensemble tree splits.
- **vs Gradient Boosting:** RF achieves F1 macro 0.5187 vs GB's 0.5092. GB is theoretically more powerful but more sensitive to hyperparameter choices. With default parameters on this 132K-row dataset, RF generalizes better to the test period.

### Why all features over SelectKBest

Feature selection **decreased** F1 macro for every model. The dropped features contributed predictive power that mutual information scoring underestimated — including order-related features (`prior_order_count`, `max_prior_riskscore`, `mean_prior_riskscore`, `prior_overdue_order_count`, `prior_unresolved_order_count`, `distinct_directive_count`) and device/type attributes (`device_type_encoded`, `insp_type_Other`, `last_inspection_outcome`). Using all 17 features is the correct choice for this dataset.